# S&P 500 Monthly Returns Analysis — 2025

In this notebook I'll analyze the monthly performance of the S&P 500 index during 2025.  
This is a great dataset for NumPy because financial analysis is built on vectorized arithmetic — returns, risk, compounding.

**Source:** https://www.statmuse.com/money/ask/s-and-p-500-monthly-returns-for-2025
*(Monthly % change calculated from end-of-month closing prices)*

---

In [2]:
import numpy as np


In [3]:
# --- Data ---
months = np.array(['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec'])

# Monthly returns (%) — S&P 500, 2023
returns = np.array([
     2.33,   # Jan
    -0.25,   # Feb
     5.97,   # Mar
    -0.51,   # Apr
     5.09,   # May
     5.23,   # Jun
     2.46,   # Jul
     2.75,   # Aug
     4.48,   # Sep
     2.63,   # Oct
    -0.48,   # Nov
     0.49,   # Dec
])

---
## Part 1 — Basic Performance

Let's start with the fundamentals — what did 2023 look like at a glance?

In [8]:
# Let's find the best and worst month (name + value),
print(f"Best month {months[returns.argmax()]} : {returns.max()}")
print(f"Worst month {months[returns.argmin()]} : {returns.min()}")

# the average monthly return, and the simple sum of all returns
print(f"Average monthly return: {returns.mean():.2f}")
print(f"Sum of all returns: {returns.sum():.2f}")

Best month Mar : 5.97
Worst month Apr : -0.51
Average monthly return: 2.52
Sum of all returns: 30.19


In [16]:
# Now let's split the year into quarters and compare them
# Q1 = Jan-Mar, Q2 = Apr-Jun, Q3 = Jul-Sep, Q4 = Oct-Dec
q1 = returns[0:3]
q2 = returns[3:6]
q3 = returns[6:9]
q4 = returns[9:12]
quarters = np.array([q1.sum(),q2.sum(),q3.sum(),q4.sum()])
q_names = np.array(['Q1','Q2','Q3','Q4'])

for n,q in zip(q_names,quarters):
    print(f"{n}: {q:.2f}")

# Which quarter was the best? Which was the worst?
print(f"Best quarter: {q_names[quarters.argmax()]}")
print(f"Worsr quarter: {q_names[quarters.argmin()]}")


Q1: 8.05
Q2: 9.81
Q3: 9.69
Q4: 2.64
Best quarter: Q2
Worsr quarter: Q4


---
## Part 2 — Risk Metrics

Return alone doesn't tell the full story. A 10% return with extreme volatility is very different from a steady 10%.  
Let's quantify the risk side of 2023.

In [21]:
# Calculate:
# - Standard deviation (this IS volatility in finance)
print(f"Standart Deviation: {returns.std():.2f}")

# - Variance
print(f"Variance: {returns.var():.2f}")

# - A simplified return/risk ratio: mean / std
#   (higher = better risk-adjusted performance)

print(f"Return risk ratio: {(returns.mean()/returns.std()):.2f}")


Standart Deviation: 2.23
Variance: 4.98
Return risk ratio: 1.13


In [25]:
# Let's also calculate the downside deviation —
# standard deviation of ONLY the negative months
negative_months= returns[returns<0]
print(f"Downside Deviation: {(negative_months.std()):.2f}")

# This is a more conservative risk measure used in finance


Downside Deviation: 0.12


---
## Part 3 — Up vs Down Months

Let's use boolean indexing to separate winning and losing months.

In [33]:
# Separate positive and negative months
positive_months = returns[returns > 0]
negative_months= returns[returns < 0]

# Print: count, average gain/loss, and win rate as percentage
print(f"Number of positive months: {len(positive_months)} | Average gain/loss: +{positive_months.mean():.2f}")
print(f"Number of negative months: {len(negative_months)} | Average gain/loss: {negative_months.mean():.2f}")

print(f"Win rate: {(len(positive_months)/len(returns)*100):.2f}%")

Number of positive months: 9 | Average gain/loss: +3.49
Number of negative months: 3 | Average gain/loss: -0.41
Win rate: 75.00%


In [34]:
# Now let's find the months where the return was above 1 standard deviation
# from the mean (in either direction) — these are the 'outlier' months
# Outlier condition: abs(return - mean) > std
mean = returns.mean()
std = returns.std()
condition = np.abs(returns - mean) > std
print(f"Outlier months: {months[condition]}")





Outlier months: ['Feb' 'Mar' 'Apr' 'May' 'Jun' 'Nov']


---
## Part 4 — Compounding

This is where things get interesting. Summing percentages is NOT the same as compounding them.  
Let's see the difference.

In [ ]:
# Calculate the compounded annual return:
# Each month: multiplier = 1 + (return / 100)
# Total: multiply all multipliers together (np.prod), subtract 1, multiply by 100
# Compare against the simple sum and print the difference
multipliers = 1 + returns/100
total = (multipliers.prod() - 1)*100
print(f"Compound annual return: {total:.2f}")
print(f"Annual simple sum: {returns.sum():.2f}")
print(f"Difference: {(total - returns.sum()):.2f}")


Compound annual return: 34.36
Annual simple sum: 30.19
Difference: 4.17


In [43]:
# Now simulate how $10,000 invested at the start of 2023 would have grown month by month
# Use np.cumprod() to get the cumulative multiplier at each month
# Print the portfolio value at the end of each month

portfolio = 10000 * (multipliers.cumprod())
for m,p in zip(months, portfolio):
    print(f"{m}: {p:.2f}")

Jan: 10233.00
Feb: 10207.42
Mar: 10816.80
Apr: 10761.63
May: 11309.40
Jun: 11900.88
Jul: 12193.65
Aug: 12528.97
Sep: 13090.27
Oct: 13434.54
Nov: 13370.06
Dec: 13435.57


---
## Part 5 — Going Further

A couple of extra ideas to push the analysis.

In [ ]:
# Calculate the rolling 3-month average return using a loop + slicing
# (NumPy has np.convolve for this but let's do it manually first to understand it)
# For each window of 3 consecutive months, calculate the average
# This smooths out noise and shows the underlying trend


for i in range(len(returns)-2):
    window = returns[i:i+3]
    print(f"{months[i]} - {months[i+2]} = {window.mean():.2f}")

Jan - Mar = 2.68
Feb - Apr = 1.74
Mar - May = 3.52
Apr - Jun = 3.27
May - Jul = 4.26
Jun - Aug = 3.48
Jul - Sep = 3.23
Aug - Oct = 3.29
Sep - Nov = 2.21
Oct - Dec = 0.88


In [52]:
# Use np.where() to create a label array:
# 'bull' if return > 0, 'bear' if return <= 0
# Then count consecutive bear months (months where the market was down back to back)

label = np.where(returns > 0, 'bull', 'bear')
print(f"Market labels: {label}")

max_streak, current = 0, 0
for r in returns:
    if r < 0:
        current += 1
        max_streak = max(max_streak, current)
    else:
        current = 0
print(f"Longest bear streak: {max_streak} months")

Market labels: ['bull' 'bear' 'bull' 'bear' 'bull' 'bull' 'bull' 'bull' 'bull' 'bull'
 'bear' 'bull']
Longest bear streak: 1 months


---
## Observations

*Write your findings here after completing the analysis.*

- The S&P had a strong year with 9 positive months out of 12
- March was the strongest month, recovering from the worst quater Q4
- The compounding effect adds meaningful return over the simple sum 